# **Clasificación de universidades: Pública o privada**

En este notebook llevamos a cabo la clasifiación de universidades si son públicas o privadas para posteriores análisis. Si es cierto que al ser del BOE no se nombran muchas universidades privadas.

In [1]:
!pip install unidecode


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 8.2 MB/s eta 0:00:00


Creamos listas manualmente para identificar qué universidades son públicas o privadas.

In [4]:
# ----------------- UNIVERSIDADES PÚBLICAS -----------------
universidades_publicas = list(set([
    "Universidad de Alcalá", "Universidad de Alicante", "Universidad de Almería",
    "Universidad de Cádiz", "Universidad de Castilla-La Mancha", "Universidad Complutense de Madrid",
    "Universidad de Córdoba", "Universidad de Extremadura", "Universidad de Granada",
    "Universidad de Huelva", "Universidad de Jaén", "Universidad de La Laguna",
    "Universidad de La Rioja", "Universidad de Las Palmas de Gran Canaria",
    "Universidad de León", "Universidad de Málaga", "Universidad de Murcia",
    "Universidad de Oviedo", "Universidad de Salamanca", "Universidad de Sevilla",
    "Universidad de Valladolid", "Universidad de Zaragoza", "Universidad del País Vasco",
    "Universidad de Cantabria", "Universidad Carlos III de Madrid",
    "Universidad Autónoma de Madrid", "Universidad Autónoma de Barcelona",
    "Universidad de Valencia", "Universidad Politécnica de Valencia",
    "Universidad Politécnica de Madrid", "Universidad Politécnica de Cataluña",
    "Universidad de Santiago de Compostela", "Universidad de Vigo",
    "Universidad de A Coruña", "Universidad Nacional de Educación a Distancia",
    "Universidad Rey Juan Carlos", "Universidad Pablo de Olavide",
    "Universidad de Burgos", "Universidad de Lleida", "Universidad de Girona",
    "Universidad de Vic - Central de Cataluña", "Universidad de Barcelona",
    "Universidad Rovira i Virgili", "Universidad Miguel Hernández de Elche",
    "Universidad Pública de Navarra", "Universidad Internacional Menéndez Pelayo",
    "Universidad Jaume I", "Universidad de las Islas Baleares",
    "Universidad Politécnica de Cartagena", "Universidad Pompeu Fabra",
    "Universidad Internacional de Andalucía", "Universidad San Jorge",
    "Universidad las Illes Balears", "Universidad Miguel Hernández",
    "Universidad «Rovira i Virgili» de Tarragona", "Universidad de las Illes Balears",'Universidad Cumplutense de Madrid',
    'Universitat Politècnica de València', 'Universidad Politécnica de Almería', 'Universidad Complutense',
    'Universidad de Alcálá de Henares', "Universidad del Pais Vasco", "Universidad «Carlos III»",
    'Universidad de Códoba', 'Universidad Politécnica de Barcelona', "Universidad de Vic",
    "Universidad «Miguel Hernández»", "Universitat Autònoma de Barcelona", "Universidad «Rey Juan Carlos»",
    "Universidad Las Palmas de Gran Canaria", "País Vasco/Euskal Herriko Unibertsitatea",
    "Universidad de las «Illes Balears»", "Escuela Universitaria Politécnica de Donostia-San Sebastián",
    "Universidad de Traducción e Interpretación de Granada", "Instituto Nacional de Educación Física de Catalunya",
    "Escuela Universitaria de Fisioterapia de A Coruña", "Universidad Illes Balears",
    "Universidad «Pablo de Olavide»", "Universidad de Granaada","Universidad «Rovira i Virgili",
    "Universidad Santiago de Compostela",
    "Universitat Politècnica de Catalunya",
    "Universidad Politècnica de Catalunya",
    "Universidade de A Coruña",
    "Universidad de Vic-Universidad Central de Catalunya",
    "Universidad Central de Catalunya",
    "Universidad Rovira y Virgili",
    "Universidad Oberta de Catalunya",
    "Universitat Jaume I", "Universidad «Jaume i»"
]))


# ----------------- UNIVERSIDADES PRIVADAS -----------------
universidades_privadas = list(set([
    "Universidad de Navarra", "Universidad Pontificia Comillas",
    "Universidad San Pablo CEU", "Universidad Francisco de Vitoria",
    "Universidad Europea de Madrid", "Universidad Camilo José Cela",
    "Universidad Internacional de Cataluña", "Universidad Católica de Ávila",
    "Universidad Católica San Antonio de Murcia", "Universidad Nebrija",
    "Universidad Alfonso X el Sabio", "Universidad Abat Oliba CEU",
    "Universidad CEU Cardenal Herrera", "Universidad Villanueva",
    "Universidad a Distancia de Madrid", "Universidad Internacional de La Rioja",
    "Universidad Europea del Atlántico", "Universidad Loyola Andalucía",
    "Universidad Internacional Isabel I", "Universidad Europea de Valencia",
    "Universidad de Deusto", "Universidad Europea Miguel de Cervantes",
    "IE Universidad", "Universidad Pontificia de Salamanca",
    "Universidad Internacional Valenciana (VIU)",
    "Universidad Fernando Pessoa", "Universidad Ramon Llull",
    "Universidad Católica", "Universidad Católica Santa Teresa de Jesús",
    "Universidad Católica de Valencia San Vicente Mártir",
    "Universidad Católica San Antonio", "Universidad Antonio de Nebrija",
    "Universitat Oberta de Catalunya", "Universitat Abat Oliba CEU", 'Universidad Cardenal Herrera-CEU', 'Universidad San Pablo-CEU',
    'Universidad Oberta de Cataluña', 'Universidad de Mondragón', "Universidad «Ramón Llull»", 'Universitat Internacional de Catalunya',
    'Universidad Europea de Canarias', 'Universidad S.E.K.', "Universidad Internacional de la Empresa",
    "Escuela Universitaria de Enfermería Ntra. Sra. Desamparados",
    "Centro de Estudios Superiores Sociales y Jurídicos Ramón Carande",
    "Centro de Enseñanza Superior en Humanidades y Ciencias de la Educación Don Bosco",
   "Universidad Cardenal Herrera-C.E.U", "Universidad «Pompeu Fabra»", "Universidad Internacional Villanueva",
    "Universidad Pontificia de Comillas"
]))





Identificamos las entidades de universidades y comprobamos si el nombre detectado está en las listas definidas. Según su presencia ponemos un 1 en la columna privada o publica, la que corresponda.

Además, hemos añadido unos diccionarios con alias y universidades escritas en catalán o en gallego para que las sustituya por el nombre en castellano que coincide con los escritos en las listas.

Lo procesamos por lotes y con multiprocesamiento para que no tarde tanto.

In [5]:
import pandas as pd
import spacy
from spacy.matcher import PhraseMatcher
import re
import unidecode
from multiprocessing import Pool, cpu_count
import numpy as np


df = pd.read_csv("dataset_universidades_procesado.csv")
df['Titulo_Semantico'] = df['Titulo_Semantico'].fillna('')


# Carga el modelo spaCy optimizado (solo NER)
try:
    nlp = spacy.load("es_core_news_md", disable=["parser", "tagger", "lemmatizer"])
except:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_md"])
    nlp = spacy.load("es_core_news_md", disable=["parser", "tagger", "lemmatizer"])



# Carga siglas comunes
siglas = {
    "uned": "Universidad Nacional de Educación a Distancia",
    "upm": "Universidad Politécnica de Madrid",
    "uax": "Universidad Alfonso X el Sabio",
    "uab": "Universidad Autónoma de Barcelona",
    "uam": "Universidad Autónoma de Madrid",
    "ub": "Universidad de Barcelona",
    "us": "Universidad de Sevilla",
    "ucm": "Universidad Complutense de Madrid",
    "uv": "Universidad de Valencia",
    "urjc": "Universidad Rey Juan Carlos",
    "upv": "Universidad Politécnica de Valencia",
    "uc3m": "Universidad Carlos III de Madrid",
    "udc": "Universidad de A Coruña",
    "uoc": "Universidad Oberta de Cataluña",
    "ud": "Universidad de Deusto"
}

# Alias y variantes repetidas
alias = {
    "universidade da coruna": "universidad de a coruña",
    "universidade da coruña": "universidad de a coruña",
    "universidade de santiago de compostela": "universidad de santiago de compostela",
    "universitat politecnica": "universidad politécnica de cataluña",
    "universitat juame i": "universidad jaume i de castellón",
    "universitat de valencia": "universidad de valencia",
    "universitat de barcelona": "universidad de barcelona",
    "universitat pompeu fabra": "universidad pompeu fabra",
    "universitat autonoma de barcelona": "universidad autonoma de barcelona",
    "universitat de girona": "universidad de girona",
    "universitat de lleida": "universidad de lleida",
    "universitat de les illes balears": "universidad de las islas baleares",
    "universitat rovira i virgili": "universidad rovira i virgili",
    "universidad de la coruna": "universidad de a coruña",
    "universidad de la coruña": "universidad de a coruña",
    "universidad de les illes balears": "universidad de las islas baleares",
    "universidad carlos iii": "universidad carlos iii de madrid",
    "universidad de santiago": "universidad de santiago de compostela",
    "universidad cardenal herrera ceu": "universidad ceu cardenal herrera",
    "universidad ceu cardenal herrera": "universidad ceu cardenal herrera",
    "universidad de baleares": "universidad de las islas baleares",
    "universidad baleares": "universidad de las islas baleares",
    "universidad catolica santa teresa de jesus": "universidad catolica de avila",
    "universidad pontifica de comillas": "universidad pontificia comillas",
    "universitat oberta de catalunya": "Universidad Oberta de Catalunya",

}

# Normalización por si queda algún caracter especial
def normalizar_texto(texto):
    texto = unidecode.unidecode(texto.lower())
    texto = re.sub(r'[«»"“”]', '', texto)
    texto = re.sub(r'[-–—]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

# Precalcular estructuras normalizadas
universidades_norm = {normalizar_texto(u): u for u in universidades_publicas + universidades_privadas}
universidades_norm_simple = {normalizar_texto(u).replace(" de ", " "): u for u in universidades_publicas + universidades_privadas}
alias_norm = {normalizar_texto(k): v for k, v in alias.items()}
siglas_re = {re.compile(rf"\b{sig}\b"): nombre for sig, nombre in siglas.items()}

# Matcher exacto
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
patterns = [nlp.make_doc(u.lower()) for u in universidades_publicas + universidades_privadas]
matcher.add("UNIVERSIDADES", patterns)

def extraer_universidades_doc(doc):
    texto_norm = doc.text
    encontradas = set()

    # 1️Matcher
    for _, start, end in matcher(doc):
        encontradas.add(doc[start:end].text)

    # Búsqueda directa
    for base, original in universidades_norm.items():
        if base in texto_norm:
            encontradas.add(original)
    for base, original in universidades_norm_simple.items():
        if base in texto_norm:
            encontradas.add(original)

    # Alias
    for key, val in alias_norm.items():
        if key in texto_norm:
            encontradas.add(val)

    # Siglas
    for regex, nombre in siglas_re.items():
        if regex.search(texto_norm):
            encontradas.add(nombre)

    # NER
    for ent in doc.ents:
        if ent.label_ == "ORG" and "universidad" in ent.text.lower():
            encontradas.add(ent.text)

    return list(encontradas)

# Procesamos por lotes
def procesar_chunk(chunk):
    textos_norm = chunk['Titulo_Semantico'].astype(str).apply(normalizar_texto)
    docs = list(nlp.pipe(textos_norm, batch_size=1000))
    chunk['universidades_mencionadas'] = [extraer_universidades_doc(doc) for doc in docs]
    return chunk

# Multiproceso para acelerar
n_cores = max(1, cpu_count() - 1)
chunks = np.array_split(df, n_cores)

with Pool(n_cores) as p:
    df = pd.concat(p.map(procesar_chunk, chunks))

# Clasificación
def clasificar_universidad(lista):
    lista_l = [u.lower() for u in lista]
    es_publica = any(u in [pub.lower() for pub in universidades_publicas] for u in lista_l)
    es_privada = any(u in [priv.lower() for priv in universidades_privadas] for u in lista_l)
    return pd.Series({'publica': int(es_publica), 'privada': int(es_privada)})

df[['publica', 'privada']] = df['universidades_mencionadas'].apply(clasificar_universidad)

# Resultados
print("Conteo universidades públicas identificadas:", df['publica'].sum())
print("Conteo universidades privadas identificadas:", df['privada'].sum())
no_ident = df[df['publica'].eq(0) & df['privada'].eq(0)]
print("No identificadas:", len(no_ident))

# Muestras de casos no identificados
for i, row in no_ident.sample(10, random_state=42).iterrows():
    doc = nlp(row['Titulo_Semantico'])
    print(f"- {row['Titulo_Semantico'][:120]}...  → entidades:", [ent.text for ent in doc.ents if ent.label_ == "ORG"])


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Conteo universidades públicas identificadas: 151405
Conteo universidades privadas identificadas: 6825
No identificadas: 4456
- Resolución de la Facultad de Medicina sobre extravío de título universitario....  → entidades: ['Facultad de Medicina']
- Resolución de la Escuela Universitaria de Magisterio de Bilbao sobre extravío de titulo....  → entidades: []
- Acuerdo de 17 de mayo de 2000, del Consejo de Universidades, por el que se fijan los límites de precios por estudios con...  → entidades: ['Consejo de Universidades']
- Resolución de 21 de septiembre de 2001, de la Secretaría General del Consejo de Universidades, por la que se adscriben l...  → entidades: ['Secretaría General del Consejo de Universidades', 'Profesores de Cuerpos Docentes Universitarios']
- Resolución de 29 de marzo de 2001, de la Secretaría General del Consejo de Universidades, por la que se señalan lugar, d...  → entidades: ['Secretaría General del Consejo de Universidades', 'Comisiones', 'Cuerpos Docentes Universi

Los titulares que no sea han ejecutado es porque nombran Consorcios o consejos de universidades en su mayoría. Aunque también hay escuelas universitarias y centros adscritos en los que no se especifica a qué universidades pertenecen. Después de analizar estas disposiciones hemos identificado escuelas universitarias privadas y públicas. Así, creamos diccionarios para mapear estas escuelas: las escuelas que nombran ciudades normalmente pertenecen a las universidades públicas de esas ciudades; los consorcios públicos los establecemos como públicos; los centros universitarios privados conocidos los clasificamos como privados; y el resto de organismos los dejamos a 0 en ambas columnas.

In [7]:
mapeo_privadas = {
    "gimbernat": "privada",
    "blanquerna": "privada",
    "sek": "privada",
    "atlántico medio": "privada",
    "ceu": "privada",
    "nebrija": "privada",
    "san pablo": "privada",
    "villanueva": "privada",
}

mapeo_ciudades_publicas = {
    "bilbao": "publica",
    "mieres": "publica",
    "valencia": "publica",
    "sevilla": "publica",
    "granada": "publica",
    "santiago": "publica",
    "coruña": "publica",
    "barcelona": "publica",
    "madrid": "publica",
    "salamanca": "publica",
    "zaragoza": "publica",
    "san sebastián": "publica",
}

def reforzar_no_identificados(row):
    # Solo actua si no se clasificó antes
    if row["publica"] == 0 and row["privada"] == 0:
        texto = row["Titulo_Semantico"].lower()

        # Escuelas o facultades por ciudad
        for ciudad in mapeo_ciudades_publicas.keys():
            if ciudad in texto and any(p in texto for p in ["escuela", "facultad"]):
                row["publica"] = 1
                return row

        # Universidades privadas conocidas
        for clave in mapeo_privadas.keys():
            if clave in texto:
                row["privada"] = 1
                return row

        # Organismos administrativos
        if any(p in texto for p in [
            "consejo de universidades",
            "coordinación universitaria",
            "secretaría general del consejo"
        ]):
            # Mantiene 0,0
            return row

        # Consorcios públicos
        if any(p in texto for p in ["consorci", "consorcio", "cbuc", "csuc"]):
            row["publica"] = 1 # pública
            return row

    return row

df = df.apply(reforzar_no_identificados, axis=1)

print("Conteo universidades públicas identificadas:", df['publica'].sum())
print("Conteo universidades privadas identificadas:", df['privada'].sum())
no_ident = df[df['publica'].eq(0) & df['privada'].eq(0)]
print("No identificadas:", len(no_ident))

Conteo universidades públicas identificadas: 151997
Conteo universidades privadas identificadas: 6894
No identificadas: 3795


In [8]:
df.to_csv("boe_universidades_publica_priv.csv", index=False)